# YOLOv8 Train (Lokal) — Training asv

Train YOLOv8 dengan dataset **lokal** (hasil `tools/camera_capture.py`) tanpa hardcode Roboflow. Cocok export ONNX → dipakai Repo A/Repo B.

In [ ]:
import os
HOME = os.getcwd()
print("CWD:", HOME)
print("Dataset path contoh:", os.path.join(HOME, "dataset", "data.yaml"))

## Install ultralytics (training)
Jalankan hanya kalau belum: `pip install -r requirements-train.txt`

In [ ]:
# from ultralytics import YOLO
# !pip install ultralytics==8.2.103 -q

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display

## Opsi: dataset dari Roboflow (kalau tidak capture lokal)

Kalau dataset sudah dianotasi di Roboflow: isi `RF_*` di bawah, set `USE_ROBOFLOW = True`, lalu jalankan cell berikutnya.
Hasil unduhan (format YOLOv8) masuk ke folder `dataset/`.

Tautan/identitas dataset ada di URL dashboard: `app.roboflow.com/<workspace>/<project>/<versi>`.

**Jangan tulis API key di notebook.** Set lewat env: `export ROBOFLOW_API_KEY=xxxx`
(Windows: `setx ROBOFLOW_API_KEY xxxx`, lalu restart terminal).

> Alternatif tanpa SDK: dashboard Roboflow -> *Download Dataset* -> format **YOLOv8**,
> unduh zip lalu ekstrak isinya ke folder `dataset/`.

> Capture lokal? Lewati saja cell di bawah (`USE_ROBOFLOW = False`).

In [ ]:
# --- OPSIONAL: unduh dataset ber-anotasi dari Roboflow ---
ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "")
RF_WORKSPACE = "ardis-workspace-ljcds"   # dari URL: app.roboflow.com/<workspace>/<project>/<versi>
RF_PROJECT   = "pod_detection"
RF_VERSION   = 1
USE_ROBOFLOW = False                     # True = unduh dari cloud, False = pakai dataset/ lokal

if USE_ROBOFLOW:
    assert ROBOFLOW_API_KEY, "set env ROBOFLOW_API_KEY dulu (jangan hardcode di notebook)"
    !pip install roboflow --quiet
    from roboflow import Roboflow
    version = (Roboflow(api_key=ROBOFLOW_API_KEY)
               .workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION))
    version.download("yolov8", location="dataset")
    print("Roboflow dataset ->", version.location)
else:
    print("Lewati Roboflow - pakai dataset/ lokal hasil tools/camera_capture.py")


## Pilih model + data

In [ ]:
DATA = os.path.join(HOME, "dataset", "data.yaml")
assert os.path.exists(DATA), (
    f"data.yaml tidak ada: {DATA}\n"
    "-> capture lokal (tools/camera_capture.py) ATAU aktifkan USE_ROBOFLOW di cell atas")
MODEL = "yolov8n.pt"
EPOCHS = 50
IMGSZ = 640
print("DATA:", DATA)


## Train

In [ ]:
model = YOLO(MODEL)
results = model.train(data=DATA, epochs=EPOCHS, imgsz=IMGSZ, plots=True)
best = os.path.join(HOME, "runs", "detect", "train", "weights", "best.pt")
print("BEST:", best)

## Hasil training

In [ ]:
display(Image(filename=os.path.join(HOME, 'runs','detect','train','confusion_matrix.png'), width=600))
display(Image(filename=os.path.join(HOME, 'runs','detect','train','results.png'), width=600))
display(Image(filename=os.path.join(HOME, 'runs','detect','train','val_batch0_pred.jpg'), width=600))

## Validasi

In [ ]:
model.val(data=DATA, model=best)

## ONNX untuk kapal, PT untuk Python

Training di cell **Train** sudah otomatis menghasilkan **`runs/detect/train/weights/best.pt`** —
itu model aslinya. `best.pt` dipakai untuk `model.predict(...)`, training lanjutan, atau `yolo` CLI.

Masalahnya `best.pt` **tidak bisa dibaca `cv2.dnn`** (yang dipakai Repo A/Repo B). Karena itu bobot
yang sama diekspor ke **ONNX**: isinya identik, cuma formatnya bisa dibuka OpenCV.

| Ingin | Yang dilakukan |
|---|---|
| Pakai `.pt` saja | Lewati cell di bawah — `best.pt` sudah jadi |
| Dipakai kapal (Repo A/B, `cv2.dnn`) | Jalankan cell di bawah (ekspor ONNX) |
| Dua-duanya | Jalankan semua — dapat `best.pt` **dan** `best.onnx` |

Jadi kalau cuma butuh `.pt`, **tidak perlu mengubah apa pun** — cukup jangan jalankan cell ekspor ini.

In [ ]:
from shutil import copy2

# best.pt SUDAH ada hasil training — tidak dibuat ulang di sini
print("PT  :", best)

# ekspor ke ONNX (format yang bisa dibaca cv2.dnn di Repo A/Repo B)
onnx_out = model.export(format="onnx", opset=12, simplify=True)
print("ONNX:", onnx_out)

# kumpulkan hasilnya di weights/ biar gampang dicopy ke repo kapal
os.makedirs(os.path.join(HOME, "weights"), exist_ok=True)
dst = os.path.join(HOME, "weights", os.path.basename(str(onnx_out)))
copy2(str(onnx_out), dst)
print("disalin ke:", dst)
